In [1]:
%%capture
!pip install facenet-pytorch

In [2]:
import sys
sys.path.append('/home/pj00/projects/Github/small_face_recognition_trcking/utils')

## Libraries

In [41]:
import numpy as np
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training

import torch
import torch.nn as nn
from torch.utils.data import Subset

from torchvision import models
from torchvision import transforms
from torchsummary import summary
from PIL import Image

from image_iter import FaceDataset, customSubset
from custom_model import distill_model, distill_model_2
from utils import model_size

import pickle
from tqdm import tqdm

## Load dataset

In [42]:
train_root = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec'

dataset = FaceDataset(path_imgrec=train_root, rand_mirror=True)

BATCH_SIZE = 32
trainloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec /home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.idx
header0 label [490624. 501196.]
id2range 10572


In [43]:
ss = customSubset(train_root)
indices = []
for i in range(1000, 2000):
    indices += ss.class_dict[i]
    
dataset_sub = Subset(dataset, indices)
    
trainloader = torch.utils.data.DataLoader(dataset_sub, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

## Load Models

In [44]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=0)

In [45]:
teacher = InceptionResnetV1(pretrained='casia-webface').to(device)
student = distill_model().to(device)
weight_path = '/home/pj00/projects/Github/small_face_recognition_trcking/model-weights/mobV3_adam_29.pt'
student.load_state_dict(torch.load(weight_path))

<All keys matched successfully>

## Evaluate the model

In [46]:
teacher.eval()
student.eval()
print('Done')

Done


In [47]:
mse_loss = []
cosine_sim = []

for images, _ in tqdm(trainloader):
    z1 = teacher(x.to(device, dtype=torch.float32))
    z2 = student(x.to(device, dtype=torch.float32))
    
    with torch.no_grad():
        mse = torch.mean(torch.square(z1-z2), axis=1)
        cs = torch.nn.functional.cosine_similarity(z1, z2)

        mse_loss += list(mse.detach().cpu().numpy())
        cosine_sim += list(cs.detach().cpu().numpy())

100%|███████████████████████████████████| 1767/1767 [02:28<00:00, 11.87it/s]


In [49]:
print('Mean MSE: {}'.format(np.mean(mse_loss)))
print('Mean Cosine Similarity: {}'.format(np.mean(cosine_sim)))

Mean MSE: 5.282081019686302e-06
Mean Cosine Similarity: 0.9986498951911926
